In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

cleaned_path = Path("/content/hr_employee_attrition_cleaned.csv")
df = pd.read_csv(cleaned_path)

if "Attrition_Flag" not in df.columns:
    if "Attrition" in df.columns:
        df["Attrition_Flag"] = df["Attrition"].map({"Yes": 1, "No": 0})
    else:
        raise ValueError("Attrition_Flag or Attrition column is required")

output_dir = Path("reports")
output_dir.mkdir(parents=True, exist_ok=True)

if "Department" in df.columns:
    attr_by_dept = (
        df.groupby("Department")["Attrition_Flag"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "AttritionRate", "count": "EmployeeCount"})
    )
    attr_by_dept.to_csv(output_dir / "attrition_by_department.csv", index=False)

if "JobRole" in df.columns:
    attr_by_role = (
        df.groupby("JobRole")["Attrition_Flag"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "AttritionRate", "count": "EmployeeCount"})
    )
    attr_by_role.to_csv(output_dir / "attrition_by_jobrole.csv", index=False)

if "OverTime" in df.columns:
    attr_by_overtime = (
        df.groupby("OverTime")["Attrition_Flag"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "AttritionRate", "count": "EmployeeCount"})
    )
    attr_by_overtime.to_csv(output_dir / "attrition_by_overtime.csv", index=False)

if "MonthlyIncome" in df.columns:
    bins = [0, 3000, 6000, 9000, 12000, df["MonthlyIncome"].max() + 1]
    labels = ["VeryLow", "Low", "Medium", "High", "VeryHigh"]
    df["IncomeBand"] = pd.cut(df["MonthlyIncome"], bins=bins, labels=labels, include_lowest=True)
    attr_by_income = (
        df.groupby("IncomeBand")["Attrition_Flag"]
        .agg(["mean", "count"])
        .reset_index()
        .rename(columns={"mean": "AttritionRate", "count": "EmployeeCount"})
    )
    attr_by_income.to_csv(output_dir / "attrition_by_income_band.csv", index=False)

satisfaction_cols = []
for col in df.columns:
    if "Satisfaction" in col and df[col].dtype != "object":
        satisfaction_cols.append(col)

satisfaction_summary_list = []
for col in satisfaction_cols:
    grouped = (
        df.groupby("Attrition_Flag")[col]
        .mean()
        .reset_index()
        .rename(columns={col: "MeanScore"})
    )
    grouped["Metric"] = col
    satisfaction_summary_list.append(grouped)

if satisfaction_summary_list:
    satisfaction_summary = pd.concat(satisfaction_summary_list, ignore_index=True)
    satisfaction_summary.to_csv(output_dir / "satisfaction_by_attrition.csv", index=False)


/tmp/ipython-input-1723534627.py:49: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("IncomeBand")["Attrition_Flag"]
